# COLMAP vs DA3 and 3D Gaussian Splatting

This notebook walks through the full pipeline from a set of monocular images to a trained 3D Gaussian Splatting (3DGS) model. The first route uses a classical COLMAP sparse reconstruction. The second route uses DA3-Streaming depth and pose estimates. Both sections write intermediate files into a shared output folder so the full workflow can run in one Colab runtime without Google Drive.

1. **Structure-from-Motion (SfM) with COLMAP** — a classical geometry pipeline that figures out *where* each camera was and what the scene looks like as a sparse cloud of 3D points.
2. **3D Gaussian Splatting** — a neural rendering method that takes those sparse points as a starting guess and optimizes a set of 3D Gaussians until they reproduce the original photographs when rendered.

Understanding how these two stages connect is the core lesson here. COLMAP does not produce a finished 3D model; it produces *initialization data* — camera poses and a coarse point cloud — that 3DGS then refines. If the initialization is weak, the Gaussian optimization starts in a poor part of parameter space, and render quality suffers before any neural learning even begins.

**Why endoscopy?**
The dataset comes from the [SCARED benchmark](https://arxiv.org/abs/2101.01133), which contains stereo video from a laparoscopic camera inside an abdominal cavity. Endoscopy is a demanding setting for SfM: tissue surfaces are smooth and textureless, lighting moves with the camera, and there is no sky, no rigid background, and no wide-baseline parallax to anchor scale. Failures that would only appear at the edges in outdoor scenes become central here.

**Learning goals:**
- Treat COLMAP as a set of inspectable intermediate data products, not a black box.
- Develop intuition for how *frame overlap* (the fraction of the scene shared between consecutive images) determines reconstruction quality.
- Connect sparse SfM geometry to the later quality of neural rendering.
- Practice **prediction before execution**: commit to a hypothesis, run the experiment, then reconcile.


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/FelTris/camp2_3drecon.git"
REPO_NAME = "camp2_3drecon"


def running_in_colab():
    try:
        import google.colab  # noqa: F401
    except ModuleNotFoundError:
        return False
    return True


if running_in_colab():
    repo = Path("/content") / REPO_NAME
    if not (repo / ".git").exists():
        if repo.exists():
            raise FileExistsError(f"{repo} exists but is not a Git checkout. Remove it or choose another path.")
        subprocess.run(["git", "clone", REPO_URL, str(repo)], check=True)
    os.chdir(repo)
else:
    repo = Path.cwd()
    if repo.name == "notebooks":
        repo = repo.parent

REPO = repo.resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

OUTPUT_ROOT = Path("/content/camp2_outputs") if running_in_colab() else REPO / "outputs"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

mpl_config = OUTPUT_ROOT / ".mplconfig"
mpl_config.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(mpl_config))

print("Repository:", REPO)
print("Output root:", OUTPUT_ROOT)

## Install Dependencies

In Colab, run this cell before importing the reconstruction libraries. It installs the Python requirements and clones the external model repositories used by the notebooks. After a successful install, the cell writes a small marker file in Colab's temporary output folder and restarts the runtime once so upgraded packages such as NumPy are loaded cleanly. When you rerun the notebook in the same runtime storage, the marker file makes the install step skip.

Local users can leave `INSTALL_DEPENDENCIES = False` if the environment is already set up.


In [ ]:
RESTART_AFTER_INSTALL = running_in_colab()
INSTALL_DEPENDENCIES = running_in_colab()
INSTALL_SENTINEL = OUTPUT_ROOT / ".reconstruction_stack_installed"
DA3_STREAMING_DIR = REPO / "externals" / "Depth-Anything-3" / "da3_streaming"
REQUIRED_DEPENDENCY_PATHS = [
    DA3_STREAMING_DIR,
    DA3_STREAMING_DIR / "configs" / "base_config.yaml",
]


def run_streamed(command, cwd):
    env = os.environ.copy()
    env.setdefault("PYTHONUNBUFFERED", "1")
    env.setdefault("PIP_PROGRESS_BAR", "off")
    print("$ " + " ".join(command), flush=True)
    process = subprocess.Popen(
        command,
        cwd=cwd,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="", flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)


def missing_dependency_paths():
    return [path for path in REQUIRED_DEPENDENCY_PATHS if not path.exists()]


missing_paths = missing_dependency_paths()
needs_install = INSTALL_DEPENDENCIES and (not INSTALL_SENTINEL.exists() or bool(missing_paths))

if needs_install:
    if INSTALL_SENTINEL.exists() and missing_paths:
        print("Dependency marker exists, but required DA3 files are missing:", flush=True)
        for path in missing_paths:
            print(f"  - {path}", flush=True)
        print("Rerunning the dependency installer.", flush=True)
    run_streamed(["bash", "scripts/install_reconstruction_stack.sh"], cwd=REPO)
    missing_paths = missing_dependency_paths()
    if missing_paths:
        missing_text = "\n".join(f"  - {path}" for path in missing_paths)
        raise FileNotFoundError(f"Installer finished, but required dependency paths are still missing:\n{missing_text}")
    INSTALL_SENTINEL.write_text("ok\n")
    print("\nDependency install completed successfully.", flush=True)
    if RESTART_AFTER_INSTALL:
        print("Restarting the Colab runtime once so updated packages are imported cleanly.", flush=True)
        print("After Colab reconnects, rerun the notebook from the top; this install cell will skip.", flush=True)
        try:
            from google.colab import runtime

            runtime.restart_runtime()
            raise SystemExit("Restarting Colab runtime...")
        except Exception:
            os.kill(os.getpid(), 9)
    else:
        print("Restart your Python kernel before importing the reconstruction libraries.", flush=True)
elif INSTALL_DEPENDENCIES:
    print(f"Using existing dependency install marker: {INSTALL_SENTINEL}")
    print("Required external dependency paths are present.")
    print("Delete this file and rerun the cell if you need to reinstall dependencies.")
else:
    print("Skipping dependency install outside Colab.")
    missing_paths = missing_dependency_paths()
    if missing_paths:
        print("Missing external dependency paths:")
        for path in missing_paths:
            print(f"  - {path}")
    print("Set INSTALL_DEPENDENCIES = True and rerun this cell if your local environment is not set up.")

In [ ]:
import shutil
import time
from pathlib import Path

import numpy as np
import pandas as pd
import pycolmap
import torch
from torch.utils.data import DataLoader

from camp3d.data import ScaredSubset
from camp3d.metrics import image_metrics
from camp3d.splats import (
    GaussianSplatModel,
    PosedImage,
    PosedImageDataset,
    interpolate_camera_path,
    make_gaussian_optimizers,
    render_posed_image,
    render_virtual_camera,
    train_one_epoch,
)
from camp3d.viz import read_rgb, save_render_gif, show_colmap_reconstruction_plotly, show_render_comparison, show_stereo_pair
from IPython.display import Image as DisplayImage

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

dataset = ScaredSubset(REPO / "scared_640")
print(dataset.validate())

## Start With The Images

Before running any pipeline, it pays to look at the raw data. Two things are immediately apparent in a stereo endoscopy frame:

**Texture.** Biological tissue has a characteristic matte appearance with subtle specular highlights from the light source mounted on the camera. The surface is largely smooth and lacks the strong, repeatable corners and edges that classical feature detectors (SIFT, SURF, ORB) rely on. This means COLMAP will find fewer features per image than it would on a textured outdoor scene, and those features will be noisier.

**The stereo pair.** This notebook uses only the *left* stream for reconstruction — a deliberate simplification to isolate the effect of frame overlap before adding the complexity of stereo constraints. Notebook 03 revisits the scene with both streams, which provides a known, calibrated baseline between cameras and makes scale recovery exact. For now, the right image is shown purely as a reminder that the dataset is richer than what we exploit here.


In [ ]:
left_path, right_path = dataset.stereo_pairs()[0]
show_stereo_pair(read_rgb(left_path), read_rgb(right_path))

## Prediction: What Will Frame Skipping Do?

This is a forced-prediction exercise. Write down your answers before running the next cell. The goal is not to be right — it is to surface the mental model you are currently using so you can update it against the actual measurements.

**Background: what does frame overlap do for COLMAP?**

COLMAP matches features across image pairs and then *triangulates* 3D points from matched pairs that see the same scene point from different angles. For triangulation to work, two conditions must hold:

1. **Matching**: the same physical point must appear in both images and be described by a similar feature vector.
2. **Baseline**: the two cameras must be far enough apart that the angular difference (parallax) is measurable, but close enough that the appearance has not changed too much.

When you take every 5th or every 10th frame from a video, you increase the baseline between consecutive pairs — which helps triangulation depth precision — but you also reduce overlap, which reduces the probability of a successful feature match and risks breaking the connectivity of the reconstruction graph. In the extreme case, COLMAP's incremental mapper cannot register a new image because it shares too few points with the already-reconstructed model.

**Specific failure modes to anticipate:**

- *Feature matching fails first* when the scene has changed so much between frames that descriptors do not match, even though the geometric baseline would have been fine.
- *Camera registration fails* when enough matches exist to compute a two-view geometry, but not enough inliers pass the RANSAC verification for PnP (Perspective-n-Point) registration.
- *Sparse point density drops* when cameras register successfully but the wide baseline means fewer image pairs share a given point, so fewer points survive triangulation.

Write down which of these you expect to dominate at 5× and 10× skipping, and why.


In [ ]:
image_paths = dataset.left_images()
colmap_use_gpu = torch.cuda.is_available()
colmap_gpu_index = "-1"  # "-1" lets COLMAP choose automatically; use "0" to force GPU 0.
colmap_sequential_overlap = 10  # Used only if you explicitly recompute COLMAP.
RUN_COLMAP_RECONSTRUCTION = False
experiments = {
    "all_frames": image_paths,
    "every_5th": image_paths[::5],
    "every_10th": image_paths[::10],
}

print("COLMAP GPU enabled:", colmap_use_gpu)
print("COLMAP reconstruction enabled:", RUN_COLMAP_RECONSTRUCTION)
print("COLMAP source:", "fresh runtime reconstruction" if RUN_COLMAP_RECONSTRUCTION else "bundled precomputed references")
print("Bundled COLMAP references: precomputed/01_colmap/<experiment>/sparse")
print("COLMAP matching mode if recomputing: sequential")
print("Sequential overlap if recomputing:", colmap_sequential_overlap)
for name, paths in experiments.items():
    print(f"{name:>10}: {len(paths)} images")

## COLMAP Experiment Helpers

The helper functions defined in this cell manage the bookkeeping around running three separate COLMAP reconstructions. Understanding what they do — and why — builds intuition about how SfM pipelines are structured in practice.

**The COLMAP pipeline has three distinct stages:**

1. **Feature extraction** (`pycolmap.extract_features`) — runs SIFT on every image independently and stores keypoints and descriptors in a SQLite database. This step does not know about other images; it only sees one image at a time.

2. **Sequential feature matching** (`pycolmap.match_sequential`) — matches nearby frames in image order instead of comparing every possible image pair. This is much faster for video because the frames are already ordered and adjacent frames usually overlap strongly.

3. **Incremental mapping** (`pycolmap.incremental_mapping`) — selects a good initial pair, triangulates a seed model, then alternates between registering new cameras (PnP + RANSAC) and triangulating new 3D points until no more cameras can be added. Bundle adjustment runs periodically to jointly refine cameras and points.

**Caching.** The notebook checks whether a reconstruction already exists before running COLMAP. SfM is the most expensive part of the pipeline, so skipping it on reruns saves significant time. The image manifest records the exact set of filenames used; if it changes, the cached result is deleted and COLMAP reruns from scratch.

**`choose_reconstruction`.** COLMAP's incremental mapper can produce multiple disconnected *components* — sub-models that could not be merged into a single consistent reconstruction. This helper picks the component that registered the most cameras, which is the most informative one for our purposes.

**Coordinate frames.** The `image_camtoworld` and `camera_K` helpers extract two things every renderer needs:
- The **camera-to-world transform** (a 4×4 rigid-body matrix) — where the camera was in world space and which way it was pointing.
- The **intrinsic matrix K** — the mapping from 3D camera-space points to 2D pixel coordinates, encoding focal length and principal point.

COLMAP stores poses as *camera-from-world* (the inverse convention), so `image_camtoworld` inverts the stored matrix.


In [ ]:
workspace = OUTPUT_ROOT / "01_colmap"
workspace.mkdir(parents=True, exist_ok=True)
precomputed_colmap_root = REPO / "precomputed" / "01_colmap"


def colmap_status(message):
    print(f"[{time.strftime('%H:%M:%S')}] {message}", flush=True)


def colmap_matching_config_lines():
    return [
        "matching_mode=sequential",
        f"sequential_overlap={colmap_sequential_overlap}",
    ]


def estimated_sequential_pair_count(num_images, overlap):
    return sum(min(overlap, num_images - 1 - idx) for idx in range(max(num_images - 1, 0)))


def run_sequential_matching(database_path, matching_options):
    pairing_options = pycolmap.SequentialPairingOptions(overlap=colmap_sequential_overlap)
    pycolmap.match_sequential(
        database_path,
        matching_options=matching_options,
        pairing_options=pairing_options,
    )


def prepare_image_dir(name, paths, reset=False):
    experiment_dir = workspace / name
    image_dir = experiment_dir / "images"
    manifest_path = experiment_dir / "image_manifest.txt"
    config_manifest_path = experiment_dir / "colmap_config.txt"
    image_names = [path.name for path in paths]
    config_lines = colmap_matching_config_lines()
    if reset and experiment_dir.exists():
        colmap_status(f"{name}: removing previous runtime COLMAP workspace for a fresh reconstruction")
        shutil.rmtree(experiment_dir)
    elif experiment_dir.exists() and (
        not manifest_path.exists() or manifest_path.read_text().splitlines() != image_names
        or not config_manifest_path.exists() or config_manifest_path.read_text().splitlines() != config_lines
    ):
        shutil.rmtree(experiment_dir)
    image_dir.mkdir(parents=True, exist_ok=True)
    for src in paths:
        dst = image_dir / src.name
        if not dst.exists():
            shutil.copy2(src, dst)
    manifest_path.write_text("\n".join(image_names) + "\n")
    config_manifest_path.write_text("\n".join(config_lines) + "\n")
    return experiment_dir, image_dir


def choose_reconstruction(reconstructions):
    if isinstance(reconstructions, dict):
        candidates = list(reconstructions.values())
    else:
        candidates = list(reconstructions)
    if not candidates:
        return None
    return max(candidates, key=lambda reconstruction: sum(image.has_pose for image in reconstruction.images.values()))


def run_colmap_experiment(name, paths):
    total_start = time.perf_counter()
    bundled_sparse_dir = precomputed_colmap_root / name / "sparse"
    bundled_model_dirs = sorted(path for path in bundled_sparse_dir.iterdir() if path.is_dir()) if bundled_sparse_dir.exists() else []
    if not RUN_COLMAP_RECONSTRUCTION and bundled_model_dirs:
        colmap_status(f"{name}: loading bundled COLMAP sparse model from {bundled_sparse_dir}")
        reconstruction = choose_reconstruction(pycolmap.Reconstruction(path) for path in bundled_model_dirs)
        registered = sum(image.has_pose for image in reconstruction.images.values()) if reconstruction is not None else 0
        sparse_points = len(reconstruction.points3D) if reconstruction is not None else 0
        colmap_status(
            f"{name}: loaded in {time.perf_counter() - total_start:.1f}s "
            f"({registered} registered images, {sparse_points} sparse points)"
        )
        return {"name": name, "image_dir": dataset.left_dir, "reconstruction": reconstruction}

    if not RUN_COLMAP_RECONSTRUCTION:
        raise FileNotFoundError(
            f"No bundled COLMAP sparse model found at {bundled_sparse_dir}. "
            "Set RUN_COLMAP_RECONSTRUCTION = True only if you intentionally want to recompute COLMAP."
        )

    if bundled_model_dirs:
        colmap_status(f"{name}: recompute requested; ignoring bundled model at {bundled_sparse_dir}")
    colmap_status(f"{name}: preparing {len(paths)} images for COLMAP reconstruction")
    experiment_dir, image_dir = prepare_image_dir(name, paths, reset=True)
    database_path = experiment_dir / "database.db"
    sparse_dir = experiment_dir / "sparse"
    sparse_dir.mkdir(parents=True, exist_ok=True)
    colmap_status(f"{name}: workspace is {experiment_dir}")

    model_dirs = sorted(path for path in sparse_dir.iterdir() if path.is_dir())
    if model_dirs:
        colmap_status(f"{name}: loading {len(model_dirs)} cached sparse model(s)")
        reconstruction = choose_reconstruction(pycolmap.Reconstruction(path) for path in model_dirs)
    else:
        reader_options = pycolmap.ImageReaderOptions(camera_model="OPENCV")
        extraction_options = pycolmap.FeatureExtractionOptions(
            use_gpu=colmap_use_gpu,
            gpu_index=colmap_gpu_index,
        )
        matching_options = pycolmap.FeatureMatchingOptions(
            use_gpu=colmap_use_gpu,
            gpu_index=colmap_gpu_index,
        )
        stage_start = time.perf_counter()
        colmap_status(
            f"{name}: extracting SIFT features for {len(paths)} images "
            f"(gpu={colmap_use_gpu}, gpu_index={colmap_gpu_index})"
        )
        pycolmap.extract_features(
            database_path,
            image_dir,
            reader_options=reader_options,
            extraction_options=extraction_options,
        )
        colmap_status(f"{name}: feature extraction finished in {time.perf_counter() - stage_start:.1f}s")

        stage_start = time.perf_counter()
        num_pairs = estimated_sequential_pair_count(len(paths), colmap_sequential_overlap)
        colmap_status(
            f"{name}: sequential feature matching across about {num_pairs} nearby image pairs "
            f"(overlap={colmap_sequential_overlap})"
        )
        run_sequential_matching(database_path, matching_options)
        colmap_status(f"{name}: feature matching finished in {time.perf_counter() - stage_start:.1f}s")

        stage_start = time.perf_counter()
        colmap_status(f"{name}: running incremental mapper")
        reconstruction = choose_reconstruction(
            pycolmap.incremental_mapping(database_path, image_dir, sparse_dir)
        )
        colmap_status(f"{name}: incremental mapping finished in {time.perf_counter() - stage_start:.1f}s")
        if reconstruction is None:
            colmap_status(f"{name}: no reconstruction produced after {time.perf_counter() - total_start:.1f}s")
            return {"name": name, "image_dir": image_dir, "reconstruction": None}
    registered = sum(image.has_pose for image in reconstruction.images.values()) if reconstruction is not None else 0
    sparse_points = len(reconstruction.points3D) if reconstruction is not None else 0
    colmap_status(
        f"{name}: ready in {time.perf_counter() - total_start:.1f}s "
        f"({registered} registered images, {sparse_points} sparse points)"
    )
    return {"name": name, "image_dir": image_dir, "reconstruction": reconstruction}


def image_camtoworld(image):
    cam_from_world_attr = getattr(image, "cam_from_world")
    cam_from_world = cam_from_world_attr() if callable(cam_from_world_attr) else cam_from_world_attr
    world_to_cam = np.eye(4, dtype=np.float32)
    world_to_cam[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float32)
    return np.linalg.inv(world_to_cam)


def camera_K(camera):
    return np.asarray(camera.calibration_matrix(), dtype=np.float32)


def colmap_training_data(reconstruction, image_dir):
    frames = []
    if reconstruction is None:
        return frames, np.empty((0, 3), dtype=np.float32), np.empty((0, 3), dtype=np.float32)
    for image in reconstruction.images.values():
        if not image.has_pose:
            continue
        camera = reconstruction.cameras[image.camera_id]
        frames.append(
            PosedImage(
                image_path=image_dir / image.name,
                camtoworld=image_camtoworld(image),
                K=camera_K(camera),
            )
        )

    points = []
    colors = []
    for point in reconstruction.points3D.values():
        points.append(point.xyz)
        colors.append(np.asarray(point.color) / 255.0)
    return frames, np.asarray(points, dtype=np.float32), np.asarray(colors, dtype=np.float32)


def reconstruction_summary(name, requested, reconstruction, frames, points):
    if reconstruction is None:
        return {
            "experiment": name,
            "input_images": requested,
            "registered_images": 0,
            "sparse_points": 0,
            "mean_reprojection_error": np.nan,
        }
    errors = [float(point.error) for point in reconstruction.points3D.values() if np.isfinite(point.error)]
    return {
        "experiment": name,
        "input_images": requested,
        "registered_images": len(frames),
        "sparse_points": len(points),
        "mean_reprojection_error": float(np.mean(errors)) if errors else np.nan,
    }

In [ ]:
results = {}
summary_rows = []
overall_start = time.perf_counter()
colmap_status(f"Starting {len(experiments)} COLMAP experiment(s)")

for name, paths in experiments.items():
    print(f"\nRunning or loading: {name}", flush=True)
    result = run_colmap_experiment(name, paths)
    frames, points, colors = colmap_training_data(result["reconstruction"], result["image_dir"])
    result.update({"frames": frames, "points": points, "colors": colors})
    results[name] = result
    summary_rows.append(reconstruction_summary(name, len(paths), result["reconstruction"], frames, points))

colmap_status(f"All COLMAP experiments finished in {time.perf_counter() - overall_start:.1f}s")

summary = pd.DataFrame(summary_rows)
display(summary)

## Inspect The Reconstructions

The sparse COLMAP output is not a mesh or a dense depth map — it is a set of **3D point tracks**: each point has a 3D position, a colour (averaged from the images that observed it), and a reprojection error (how far the projected point deviates from the observed 2D keypoints, in pixels).

**What the point cloud tells you:**

- *Dense regions* correspond to parts of the scene that were visible from many overlapping views and had enough texture for SIFT to find repeatable keypoints. In endoscopy this is often a specular highlight, a surgical tool edge, or a vascular marking.
- *Empty regions* do not mean the scene is empty there. They mean COLMAP could not reliably track features across that part of the scene — perhaps because the surface is textureless, or because the lighting changed between frames.
- *Camera frustums* show the registered poses. A gap in the frustum sequence means those frames were not registered — they either had too few matches to initialize, or the incremental mapper could not add them to the growing model.

**Reprojection error** is the per-point average distance (in pixels) between where COLMAP says the point should project and where the matched keypoint actually was. Values below ~1 pixel are typical for a well-calibrated system. Large errors indicate either a poor calibration model, a dynamic object that moved between frames, or a degenerate triangulation angle.

**The normalization note.** The visualization rescales everything into a unit cube so different experiments appear at a comparable scale in the viewer. The actual 3DGS training always uses the original COLMAP metric coordinates; this is purely a display convenience.


In [ ]:
def show_colmap_experiment(name):
    result = results[name]
    points = result["points"]
    frames = result["frames"]
    if len(points) == 0 or len(frames) == 0:
        print(f"{name}: no reconstruction to visualize")
        return None
    camtoworlds = np.stack([frame.camtoworld for frame in frames], axis=0)
    image_names = [frame.image_path.name for frame in frames]
    return show_colmap_reconstruction_plotly(
        points,
        result["colors"],
        camtoworlds=camtoworlds,
        image_names=image_names,
        max_points=50_000,
        point_size=2.0,
        title=f"{name}: sparse COLMAP points and cameras",
        normalize=True,
    )

### All Frames

With the full frame sequence, COLMAP has maximum overlap between consecutive images — typically 80–90% of the field of view is shared between adjacent frames. This gives the matcher plenty of opportunities to find correspondences, and the incremental mapper can usually register nearly every input image.

Expect a dense, well-connected point cloud that covers the visible surface. The camera frustums should form a smooth trajectory with no visible gaps. This is the *reference* reconstruction: any degradation you see in the 5× and 10× experiments is attributable to reduced overlap, not to the scene itself.


In [ ]:
fig = show_colmap_experiment("all_frames")
if fig is not None:
    fig.show()

### Every 5th Frame

At 5× skipping, consecutive frames in the experiment share roughly 50–70% of the field of view, depending on the speed of camera motion. This is still within the comfortable operating range of SIFT matching for most scenes, but endoscopy is not most scenes.

Look for:
- Whether COLMAP still registers most of the input images, or whether some frames are dropped.
- Whether the point cloud has visible holes compared to the all-frames case.
- Whether the mean reprojection error changes — wider baselines can actually *improve* triangulation accuracy for points that do register, because the parallax angle is larger.

The key question is whether the reconstruction remains a single connected component or breaks into fragments.


In [ ]:
if "every_5th" in results:
    fig = show_colmap_experiment("every_5th")
    if fig is not None:
        fig.show()
else:
    print("Skipping every_5th: no reconstruction available for this experiment.")

### Every 10th Frame

At 10× skipping, consecutive input frames may share less than 30% of the field of view. This is near or past the limit for reliable sequential feature matching in a textureless scene.

Two outcomes are possible depending on the specific sequence:
1. **Graceful degradation** — COLMAP still finds enough matches to maintain connectivity, but registers fewer cameras and triangulates fewer points. The point cloud is sparse and the reprojection errors may be larger.
2. **Fragmentation** — the reconstruction graph breaks into disconnected sub-models. `choose_reconstruction` will pick the largest fragment, which may cover only a portion of the scene. Some experiments may return `None` if no sub-model reaches a minimum size.

Compare these plots quantitatively against the summary table from the previous cell: `registered_images` and `sparse_points` are the clearest signals.


In [ ]:
if "every_10th" in results:
    fig = show_colmap_experiment("every_10th")
    if fig is not None:
        fig.show()
else:
    print("Skipping every_10th: no reconstruction available for this experiment.")

## Train 3DGS From The Full COLMAP Initialization

We now hand the COLMAP output — camera poses, intrinsics, and a sparse point cloud — to the 3D Gaussian Splatting optimizer. It is worth pausing to understand exactly what happens at this boundary.

**What 3DGS receives from COLMAP:**

- A set of **posed images**: each training image paired with the camera matrix K and the camera-to-world transform. These are the targets the model will try to reproduce.
- A **point cloud with colors**: each COLMAP point becomes the centre of one initial 3D Gaussian. The color of the point sets the initial colour of the Gaussian. The `init_scale` parameter controls the initial size of the Gaussians (here 0.01 scene units — small relative to the scene extent), and `init_opacity` (0.15) starts them mostly transparent.

**Why does initialization matter?**

3DGS is a differentiable renderer: it rasterizes the Gaussians into an image, computes a pixel-wise L1 loss against the target photograph, and backpropagates gradients through the rasterization to update the Gaussian parameters (mean, covariance, color, opacity). Like any gradient-based optimizer, it can get stuck in local minima if the starting configuration is too far from the ground truth.

A dense, accurate COLMAP point cloud seeds Gaussians near the true scene surface, so early renders already resemble the targets and gradients are informative. A sparse or inaccurate initialization seeds many Gaussians in empty space or the wrong location, and the optimizer must first move them to useful positions before it can start refining appearance.

**Image scaling.** The training images are loaded at 25% of their native resolution (`image_scale=0.25`). This is a common training strategy: lower-resolution images make each epoch faster and make the optimizer converge to coarse structure first. Full-resolution fine-tuning would follow in a production pipeline.


In [ ]:
baseline = results["all_frames"]
frames = baseline["frames"]
points = baseline["points"]
colors = baseline["colors"]

trainset = PosedImageDataset(frames, image_scale=0.25)
trainloader = DataLoader(trainset, batch_size=1, shuffle=True)

model = GaussianSplatModel(points, colors, init_scale=0.01, init_opacity=0.15).to(device)
optimizer = make_gaussian_optimizers(model, lr=1e-3)

print("train images:", len(trainset))
print("gaussians:", model.module.means.shape[0])

In [ ]:
for epoch in range(30):
    loss = train_one_epoch(model, trainloader, optimizer, device=device)
    print(f"epoch {epoch:02d} | L1 loss {loss:.4f}")

## Rendered Views And Image Metrics

After training, we render the scene from the original training camera poses and compare the renders to the ground-truth photographs. This is a **training-set evaluation** — we are measuring how well the model memorized the views it was trained on, not how well it generalizes to new viewpoints. Novel-view synthesis is tested in the next cell.

**PSNR (Peak Signal-to-Noise Ratio)**
PSNR is derived from per-pixel mean squared error:

$$\text{PSNR} = 10 \cdot \log_{10}\left(\frac{1}{\text{MSE}}\right)$$

Higher is better. A rough intuition: below ~20 dB the image looks noticeably blurry or wrong; above ~30 dB differences are subtle; above ~40 dB is near-lossless for natural images. Because it is based on MSE, it penalizes any pixel that is off by a lot — including small spatial misalignments — which makes it sensitive to ghosting and blurring.

**SSIM (Structural Similarity Index)**
SSIM compares patches of the image in terms of luminance (mean brightness), contrast (standard deviation), and structure (normalized correlation). It ranges from 0 to 1; values above 0.9 are generally considered good. SSIM is less sensitive to pixel-perfect alignment than PSNR and correlates better with human perceptual judgments in many benchmarks — but it is still a proxy, not a ground truth for perceived quality.

**What low metrics here actually mean.** Poor PSNR or SSIM on training views can indicate:
- The model has not converged (train for more epochs or reduce learning rate).
- The COLMAP initialization was too sparse to seed Gaussians in the right places.
- The image regions contain tissue that moved between frames (violates the static scene assumption of 3DGS).
- The camera calibration has residual error that is larger than the per-Gaussian position precision.


In [ ]:
source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
requested_sample_names = [image_paths[idx].name for idx in source_sample_ids]
frame_name_to_id = {frame.image_path.name: frame_id for frame_id, frame in enumerate(frames)}
sample_ids = [frame_name_to_id[name] for name in requested_sample_names if name in frame_name_to_id]
if not sample_ids:
    print("Requested source samples were not registered by COLMAP; using evenly spaced registered frames.")
    sample_ids = np.linspace(0, len(frames) - 1, num=min(3, len(frames)), dtype=int).tolist()
else:
    print("Rendering source samples:", requested_sample_names)
sample_manifest = workspace / "render_sample_names.txt"
sample_manifest.write_text("\n".join(frames[frame_id].image_path.name for frame_id in sample_ids) + "\n")
metric_rows = []

for frame_id in sample_ids:
    rendered = render_posed_image(model, frames[frame_id], image_scale=0.25, device=device)
    metrics = image_metrics(rendered["render"], rendered["target"])
    metric_rows.append({"frame": frames[frame_id].image_path.name, **metrics})
    show_render_comparison(
        rendered["target"],
        rendered["render"],
        rendered["alpha"],
        metrics=metrics,
    )

display(pd.DataFrame(metric_rows))

## Render An Interpolated COLMAP Trajectory GIF

Rendering the exact training views tells us about reconstruction quality, but the signature capability of 3DGS is **novel-view synthesis** — rendering the scene from camera positions that were never in the training set.

**How camera path interpolation works.**
The anchor frames (taken from the training set) define keyframe poses. Between any two keyframes, we interpolate:
- **Translation**: linear interpolation between camera centres gives a smooth dolly path.
- **Rotation**: spherical linear interpolation (SLERP) on the rotation quaternion gives a smooth, constant-speed rotation — linear interpolation on rotation matrices does not produce constant angular velocity and can distort intermediate poses.

The interpolated cameras are *novel* in the sense that they were never seen during training. If the Gaussians have been placed and shaped correctly, they should still rasterize to a plausible image from these intermediate poses. If the model has overfit to the training views or has holes in coverage, the novel views will show artifacts: floaters (stray Gaussians), holes (regions with no Gaussian coverage), or blurring (Gaussians that are too large and smear across the novel viewpoint).

**Connection to the NeRF/3DGS research community.** This interpolated-path evaluation is the standard way that papers like the original [3DGS paper (Kerbl et al. 2023)](https://repo-sam.inria.fr/fungraph/3d-gaussian-splatting/) and the Nerfstudio framework (`ns-render interpolate`) demonstrate results. The GIF you produce here is the same kind of artifact, just at smaller scale.


In [ ]:
sample_manifest = workspace / "render_sample_names.txt"
frame_name_to_id = {frame.image_path.name: frame_id for frame_id, frame in enumerate(frames)}

if sample_manifest.exists():
    anchor_names = [name for name in sample_manifest.read_text().splitlines() if name]
    print("Loaded trajectory anchors from manifest:", anchor_names)
else:
    source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
    anchor_names = [image_paths[idx].name for idx in source_sample_ids]
    print("No render sample manifest found; using deterministic source anchors:", anchor_names)

anchor_frame_ids = [frame_name_to_id[name] for name in anchor_names if name in frame_name_to_id]
if len(anchor_frame_ids) < 2:
    print("Manifest/source anchors did not include at least two registered COLMAP frames; using evenly spaced registered frames.")
    anchor_frame_ids = np.linspace(0, len(frames) - 1, num=min(3, len(frames)), dtype=int).tolist()

if len(anchor_frame_ids) < 2:
    raise ValueError("Need at least two registered COLMAP anchor frames for an interpolated trajectory.")

anchor_names = [frames[frame_id].image_path.name for frame_id in anchor_frame_ids]
sample_manifest.write_text("\n".join(anchor_names) + "\n")
anchor_frames = [frames[frame_id] for frame_id in anchor_frame_ids]

novel_cameras = interpolate_camera_path(anchor_frames, steps_per_segment=20, include_keyframes=False)
gif_renders = [
    render_virtual_camera(
        model,
        camera["camtoworld"],
        camera["K"],
        reference_frame=anchor_frames[0],
        image_scale=1.0,
        device=device,
    )["render"]
    for camera in novel_cameras
]
gif_path = save_render_gif(workspace / "colmap_3dgs_trajectory.gif", gif_renders, fps=8)
print("anchor frames:", [frame.image_path.name for frame in anchor_frames])
print("novel views:", len(gif_renders))
DisplayImage(filename=str(gif_path))

## Discussion

Work through these questions after inspecting the outputs above. They are designed to push beyond surface-level observation toward mechanistic understanding.

**1. Which regions become sparse points, and which regions stay empty?**
Look at the 3D scatter plot for the all-frames reconstruction. Do the points cluster on the tissue surface, on tool edges, or near specular highlights? Where are the gaps? Relate the gap locations back to the visual appearance of the images: are those regions smooth, uniformly lit, or near the image boundary?

**2. What changed when using every 5th and every 10th frame?**
Compare the `registered_images`, `sparse_points`, and `mean_reprojection_error` columns in the summary table. Did registration rate drop smoothly, or was there a threshold effect? Did reprojection error go up or down with sparser frames — and what does that tell you about the triangulation angle vs. noise trade-off?

**3. Does the 3DGS render fail where COLMAP has little geometry?**
Overlay your memory of the sparse point cloud with the rendered images. Are the poorly-rendered regions (high per-pixel error, blurry, or missing) the same regions that had few COLMAP points? If yes, this confirms the initialization bottleneck hypothesis. If no, something else is limiting render quality — possibly the number of training epochs, the learning rate schedule, or dynamic tissue.

**4. What changes in the 3DGS reconstruction when using every_5th or all frames?**
*(You need to train a second 3DGS model from the sparser initialization and compare PSNR/SSIM.)* Think through what you predict before doing the experiment: fewer seed points → fewer initial Gaussians → less scene coverage → lower PSNR? Or does the optimizer recover by splitting Gaussians into the empty regions?

**5. Which assumptions of feature-based SfM are weak in endoscopy?**
Classic SfM assumes: a static rigid scene, consistent lighting, texturally rich surfaces, and a calibrated camera with negligible lens distortion. Rank these assumptions from strongest to weakest violation in a laparoscopic procedure, and suggest one concrete technical approach that addresses the worst violation (hint: notebook 03 addresses the scale ambiguity problem using stereo; what would address the textureless surface problem?).


## DA3 Initialization on the Same Runtime Outputs

The cells below are the DA3 workflow from the original notebook 02, appended after the COLMAP workflow so Colab keeps all intermediate files in the same temporary runtime storage. The DA3 section still reads and writes normal files; it no longer depends on opening a second notebook with access to the same filesystem state.

By default, `da3_experiment = "all_frames"`, matching the COLMAP 3DGS baseline above. Change it to `"every_5th"` or `"every_10th"` if you want DA3 to use one of the bundled COLMAP frame-skip reconstructions.

## DA3 Initialization and Gaussian Splatting

In the COLMAP section above, we used **COLMAP** to estimate camera poses and a sparse 3D point cloud from image correspondences. That is the classical structure-from-motion route: find matching visual features across images, triangulate them, and use the resulting sparse geometry to initialize 3D Gaussian Splatting.

In this section, we try a different starting point. Instead of relying on sparse feature matches, we use **Depth Anything 3 Streaming (DA3-Streaming)** to predict a dense depth map and a camera trajectory. The depth maps are then backprojected into 3D points, which gives us a much denser initialization for Gaussian Splatting.

The comparison is intentionally narrow and controlled:

- the COLMAP section: sparse COLMAP points from the selected frame subset,
- this DA3 section: dense DA3 backprojected points from the same selected frame subset.

The goal is not to prove that one method is universally better. Instead, we want to build intuition for what changes when the 3DGS optimizer starts from **sparse feature-based geometry** versus **dense learned geometry**.

By the end of this DA3 section, you should be able to:

- explain the difference between sparse COLMAP geometry and dense depth-based geometry,
- inspect DA3 depth and confidence maps before trusting them,
- understand how a depth image can be backprojected into a 3D point cloud,
- train a 3D Gaussian Splatting model from the DA3 initialization,
- render a camera trajectory as a GIF,
- evaluate held camera views with PSNR and SSIM,
- reason about when dense learned depth helps and when it can introduce plausible-looking but uncertain geometry.

A useful mindset for this notebook is: **DA3 gives us more geometry, but more geometry does not automatically mean more correct geometry.**


In [ ]:
import shutil

import numpy as np
import pandas as pd
import pycolmap
import torch
from IPython.display import Image as DisplayImage
from PIL import Image
from torch.utils.data import DataLoader

from camp3d.data import ScaredSubset
from camp3d.metrics import image_metrics
from camp3d.splats import (
    GaussianSplatModel,
    PosedImage,
    PosedImageDataset,
    interpolate_camera_path,
    make_gaussian_optimizers,
    points_from_depth_prediction,
    render_posed_image,
    render_virtual_camera,
    train_one_epoch,
)
from camp3d.viz import save_render_gif, show_colmap_reconstruction_plotly, show_render_comparison

device = "cuda" if torch.cuda.is_available() else "cpu"
dataset = ScaredSubset(REPO / "scared_640")
all_image_paths = dataset.left_images()
da3_experiment = "all_frames"
experiment_strides = {
    "all_frames": 1,
    "every_5th": 5,
    "every_10th": 10,
}
image_stride = experiment_strides[da3_experiment]
image_paths = all_image_paths[::image_stride]
print("device:", device)
print("DA3 experiment:", da3_experiment)
print("frames:", len(image_paths), "of", len(all_image_paths))

## Load The Sparse COLMAP Reference

Before looking at the DA3 reconstruction, we first load the sparse COLMAP reconstruction from the COLMAP section. This gives us a reference point for comparison.

COLMAP reconstructs 3D points only where it can find and match visual features across multiple images. That usually means textured, well-observed areas are reconstructed more reliably, while smooth, reflective, blurry, or low-texture regions may be missing.

DA3 behaves differently. It predicts depth densely for every frame, including regions where COLMAP may not have enough feature matches. This can give much better coverage, but it also means some surfaces may be inferred by the model rather than directly supported by multi-view matching evidence.

The sparse reference is loaded from the matching bundled experiment folder by default. If `RUN_COLMAP_RECONSTRUCTION = True`, the freshly generated runtime model is preferred instead. For example, if:

```python
da3_experiment = "every_5th"
```

then the COLMAP reference comes from:

```text
REPO / "precomputed" / "01_colmap" / "every_5th" / "sparse"
```

The comparison here is mainly about:

- point density,
- spatial coverage,
- camera trajectory behavior,
- and how the different initialization affects 3DGS training.

The COLMAP visualization is normalized into a unit cube to make it easier to navigate interactively. This normalization is only for plotting. It does not change the original reconstruction or the coordinates used for training.


In [ ]:
def image_camtoworld(image):
    cam_from_world_attr = getattr(image, "cam_from_world")
    cam_from_world = cam_from_world_attr() if callable(cam_from_world_attr) else cam_from_world_attr
    world_to_cam = np.eye(4, dtype=np.float32)
    world_to_cam[:3, :4] = np.asarray(cam_from_world.matrix(), dtype=np.float32)
    return np.linalg.inv(world_to_cam)


def load_colmap_reference():
    runtime_sparse = OUTPUT_ROOT / "01_colmap" / da3_experiment / "sparse"
    bundled_sparse = REPO / "precomputed" / "01_colmap" / da3_experiment / "sparse"
    candidates = [runtime_sparse, bundled_sparse] if RUN_COLMAP_RECONSTRUCTION else [bundled_sparse, runtime_sparse]
    if da3_experiment == "all_frames":
        candidates.append(OUTPUT_ROOT / "01_colmap" / "sparse")
    sparse_dir = next((path for path in candidates if path.exists()), None)
    if sparse_dir is None:
        raise FileNotFoundError(
            "No bundled or runtime COLMAP reconstruction found for "
            f"{da3_experiment!r}. Check precomputed/01_colmap or run the COLMAP section above."
        )
    model_dirs = sorted(path for path in sparse_dir.iterdir() if path.is_dir())
    if not model_dirs:
        raise FileNotFoundError(f"No sparse model directories found in {sparse_dir}.")
    model_dir, reconstruction = max(
        ((path, pycolmap.Reconstruction(path)) for path in model_dirs),
        key=lambda item: sum(image.has_pose for image in item[1].images.values()),
    )
    points = []
    colors = []
    camtoworlds = []
    image_names = []
    for point in reconstruction.points3D.values():
        points.append(point.xyz)
        colors.append(np.asarray(point.color) / 255.0)
    for image in reconstruction.images.values():
        if image.has_pose:
            camtoworlds.append(image_camtoworld(image))
            image_names.append(image.name)
    return {
        "model_dir": model_dir,
        "points": np.asarray(points, dtype=np.float32),
        "colors": np.asarray(colors, dtype=np.float32),
        "camtoworlds": np.asarray(camtoworlds, dtype=np.float32),
        "image_names": image_names,
    }


colmap_ref = load_colmap_reference()
print("COLMAP model:", colmap_ref["model_dir"])
print("COLMAP registered cameras:", len(colmap_ref["camtoworlds"]))
print("COLMAP sparse points:", len(colmap_ref["points"]))

## Prepare DA3 Streaming Input

DA3-Streaming expects a folder of input frames. This cell prepares that folder from the selected image subset.

The notebook uses a small cache mechanism so that we do not unnecessarily copy files or rerun DA3 every time the notebook is opened. The `input_manifest.txt` file stores the list of images that should be present for the current experiment. If the selected frame subset changes, the manifest no longer matches and the input folder is rebuilt.

This is useful because DA3 inference can be relatively expensive compared with simply loading already computed outputs.

Set:

```python
force_rerun_da3 = True
```

when you intentionally want to delete the cached DA3 inputs and outputs and run everything again from scratch.

In normal use, keep it as:

```python
force_rerun_da3 = False
```

so the notebook reuses existing results whenever possible.


In [ ]:
stream_root = OUTPUT_ROOT / "02_da3_streaming" / da3_experiment
stream_input = stream_root / "input_frames"
stream_output = stream_root / "streaming_result"
processed_dir = stream_root / "processed_images"
manifest_path = stream_root / "input_manifest.txt"
force_rerun_da3 = False

expected_manifest = [path.name for path in image_paths]
if force_rerun_da3 or not manifest_path.exists() or manifest_path.read_text().splitlines() != expected_manifest:
    for path in (stream_input, stream_output, processed_dir):
        if path.exists():
            shutil.rmtree(path)
        path.mkdir(parents=True, exist_ok=True)
    for idx, src in enumerate(image_paths):
        shutil.copy2(src, stream_input / f"frame_{idx:05d}.png")
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    manifest_path.write_text("\n".join(expected_manifest) + "\n")
else:
    stream_input.mkdir(parents=True, exist_ok=True)
    stream_output.mkdir(parents=True, exist_ok=True)
    processed_dir.mkdir(parents=True, exist_ok=True)

print("stream input:", stream_input)

## Run Or Reuse DA3-Streaming

This section either loads cached DA3 outputs or runs DA3-Streaming if the required files are missing.

DA3 predicts three pieces of information that are important for us:

1. **Depth maps**  
   For each pixel, DA3 estimates how far the observed surface is from the camera.

2. **Confidence maps**  
   DA3 also provides a confidence value. This is not ground truth, but it is a useful hint about where the model thinks its prediction is more or less reliable.

3. **Camera trajectory**  
   DA3-Streaming estimates camera poses across the image sequence. These poses tell us where each camera is located and how it is oriented in 3D space.

We keep loop closure disabled here to make the output easier to inspect and reason about. Loop closure can be useful in larger SLAM-style systems, but it also adds another correction step. For teaching purposes, it is clearer to first look at the direct streaming output.

The important thing to remember is that DA3 is a learned model. It can produce dense, visually plausible geometry even in places where classical feature matching would struggle. That is powerful, but it also means we should inspect the result before using it as an initialization.


In [ ]:
poses_file = stream_output / "camera_poses.txt"
intrinsics_file = stream_output / "intrinsic.txt"
results_dir = stream_output / "results_output"

if poses_file.exists() and intrinsics_file.exists() and any(results_dir.glob("frame_*.npz")):
    print("Using cached DA3 outputs:", stream_output)
else:
    from huggingface_hub import snapshot_download

    DA3_STREAMING_DIR = REPO / "externals" / "Depth-Anything-3" / "da3_streaming"
    if not DA3_STREAMING_DIR.exists():
        raise ModuleNotFoundError(
            "DA3-Streaming is not installed. Run the Install Dependencies cell near the top of this notebook "
            "with INSTALL_DEPENDENCIES = True, then restart the Python runtime. "
            f"Missing path: {DA3_STREAMING_DIR}"
        )
    if str(DA3_STREAMING_DIR) not in sys.path:
        sys.path.insert(0, str(DA3_STREAMING_DIR))

    from da3_streaming import DA3_Streaming
    from loop_utils.config_utils import load_config
    from loop_utils.sim3utils import merge_ply_files

    weights_dir = Path(
        snapshot_download(
            "depth-anything/DA3-SMALL",
            allow_patterns=["config.json", "model.safetensors"],
        )
    )

    stream_config = load_config(str(DA3_STREAMING_DIR / "configs" / "base_config.yaml"))
    stream_config["Weights"]["DA3"] = str(weights_dir / "model.safetensors")
    stream_config["Weights"]["DA3_CONFIG"] = str(weights_dir / "config.json")
    stream_config["Model"]["chunk_size"] = 16
    stream_config["Model"]["overlap"] = 8
    stream_config["Model"]["loop_enable"] = False
    stream_config["Model"]["align_lib"] = "torch"
    stream_config["Model"]["delete_temp_files"] = True
    stream_config["Model"]["save_depth_conf_result"] = True
    stream_config["Model"]["save_debug_info"] = False
    stream_config["Model"]["Pointcloud_Save"]["sample_ratio"] = 0.02

    streamer = DA3_Streaming(str(stream_input), str(stream_output), stream_config)
    print(type(streamer.model))
    streamer.run()
    streamer.close()
    merge_ply_files(str(stream_output / "pcd"), str(stream_output / "pcd" / "combined_pcd.ply"))
    del streamer
    torch.cuda.empty_cache()

print("poses:", poses_file)
print("intrinsics:", intrinsics_file)

## Load And Inspect DA3 Geometry

Now we load the DA3 outputs from disk.

Each result file contains the image, the predicted depth map, and the confidence map for one frame. We also load the camera poses and camera intrinsics estimated by DA3.

The camera intrinsics describe the internal camera model, such as focal length and principal point. The camera pose describes where the camera is in the world. Together with the depth image, these are enough to convert pixels into 3D points.

The visualization below shows three views of the same frame:

- the RGB image,
- the DA3 depth map,
- the DA3 confidence map.

When inspecting this plot, try to ask:

- Do nearby and faraway structures make sense in the depth map?
- Are boundaries sharp or smeared?
- Are low-confidence regions aligned with difficult image regions, such as blur, reflections, specular highlights, textureless areas, or occlusions?
- Does the confidence map suggest that we should filter out some pixels before building the point cloud?

Confidence is not the same as correctness. A high-confidence prediction can still be wrong, and a low-confidence prediction can still be useful. But confidence is a practical signal for filtering noisy geometry before initializing 3DGS.


In [ ]:
def frame_index(path):
    return int(path.stem.split("_")[-1])


npz_paths = sorted((stream_output / "results_output").glob("frame_*.npz"), key=frame_index)
indices = [frame_index(path) for path in npz_paths]

images = []
depths = []
confs = []
processed_dir.mkdir(parents=True, exist_ok=True)
for path in npz_paths:
    data = np.load(path)
    image = data["image"]
    images.append(image)
    depths.append(data["depth"])
    confs.append(data["conf"])
    Image.fromarray(image).save(processed_dir / f"frame_{frame_index(path):05d}.png")

processed_images = np.stack(images)
depth = np.stack(depths).astype(np.float32)
confidence = np.stack(confs).astype(np.float32)

camtoworld = np.loadtxt(stream_output / "camera_poses.txt", dtype=np.float32).reshape(-1, 4, 4)
intrinsic_rows = np.loadtxt(stream_output / "intrinsic.txt", dtype=np.float32)
intrinsics = np.zeros((len(intrinsic_rows), 3, 3), dtype=np.float32)
intrinsics[:, 0, 0] = intrinsic_rows[:, 0]
intrinsics[:, 1, 1] = intrinsic_rows[:, 1]
intrinsics[:, 0, 2] = intrinsic_rows[:, 2]
intrinsics[:, 1, 2] = intrinsic_rows[:, 3]
intrinsics[:, 2, 2] = 1.0

camtoworld = camtoworld[indices]
intrinsics = intrinsics[indices]
extrinsics = np.linalg.inv(camtoworld)[:, :3, :].astype(np.float32)

print("images:", processed_images.shape)
print("depth:", depth.shape)
print("confidence:", confidence.shape)
print("camtoworld:", camtoworld.shape)
print("intrinsics:", intrinsics.shape)

In [ ]:
import matplotlib.pyplot as plt

i = len(processed_images) // 2
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].imshow(processed_images[i])
axes[0].set_title("image")
axes[1].imshow(depth[i], cmap="magma")
axes[1].set_title("DA3 depth")
axes[2].imshow(confidence[i], cmap="viridis")
axes[2].set_title("confidence")
for ax in axes:
    ax.axis("off")
plt.tight_layout()

## Backproject DA3 Depth

A depth map is still a 2D image: every pixel stores a distance value. To use it for 3D Gaussian Splatting, we need to turn it into 3D points.

This is done by **backprojection**. Conceptually, each pixel defines a ray leaving the camera. The depth value tells us how far along that ray the surface point should be. Using the camera intrinsics and the camera pose, we can place that point in the global 3D coordinate system.

Because DA3 predicts depth densely, a single frame can produce many points. Across many frames, this can quickly become a very large point cloud. We therefore apply a few practical filters:

- `confidence_percentile` removes the least confident depth predictions,
- `stride` keeps only every N-th pixel to reduce point count,
- `max_points` caps the final number of points.

These parameters are intentionally easy to change. Try making the point cloud denser or sparser and observe what happens.

Useful questions to keep in mind:

- Does a higher confidence threshold remove obvious noise?
- Does a smaller stride improve coverage or mostly add redundant points?
- Does the DA3 point cloud cover surfaces that COLMAP missed?
- Are some dense regions actually artifacts caused by incorrect depth?

The table compares the sparse COLMAP reference with the dense DA3 backprojection. The number of points alone is not a quality metric, but it helps show how different the two initializations are.


In [ ]:
confidence_percentile = 25
stride = 8
max_points = 80_000

points, colors = points_from_depth_prediction(
    images=processed_images,
    depths=depth,
    intrinsics=intrinsics,
    extrinsics=extrinsics,
    confidences=confidence,
    confidence_percentile=confidence_percentile,
    stride=stride,
    max_points=max_points,
)

comparison = pd.DataFrame(
    [
        {"method": "COLMAP sparse", "cameras": len(colmap_ref["camtoworlds"]), "points": len(colmap_ref["points"])},
        {"method": "DA3 backprojected", "cameras": len(camtoworld), "points": len(points)},
    ]
)
display(comparison)

In [ ]:
fig = show_colmap_reconstruction_plotly(
    colmap_ref["points"],
    colmap_ref["colors"],
    camtoworlds=colmap_ref["camtoworlds"],
    image_names=colmap_ref["image_names"],
    max_points=50_000,
    point_size=2.0,
    title=f"COLMAP sparse points from the COLMAP section ({da3_experiment})",
    normalize=True,
)
fig.show()

In [ ]:
image_names = [f"frame_{idx:05d}.png" for idx in indices]
fig = show_colmap_reconstruction_plotly(
    points,
    colors,
    camtoworlds=camtoworld,
    image_names=image_names,
    max_points=80_000,
    point_size=2.0,
    title=f"DA3 dense initialization points ({da3_experiment})",
)
fig.show()

## Train 3DGS From DA3 Initialization

Now we use the DA3 backprojected point cloud to initialize a 3D Gaussian Splatting model.

A 3DGS model represents the scene as many small, learnable Gaussian blobs in 3D space. Each Gaussian has properties such as position, color, scale, opacity, and shape. During training, the model renders the scene from known camera views and updates the Gaussians so that the rendered images match the real input images.

The initialization matters because 3DGS optimization is not magic. If the starting points are close to the real scene geometry, training has an easier job. If the starting points are sparse, noisy, or in the wrong locations, the optimizer may need more time or may converge to a worse solution.

The hypothesis in this notebook is:

> A denser DA3 initialization may give 3DGS a better starting point than sparse COLMAP points, especially in areas where COLMAP has little coverage.

But there is an important caveat:

> DA3 depth is learned and can hallucinate plausible surfaces. Dense does not always mean correct.

So when looking at the final renders, we should not only ask whether they look sharp. We should also ask whether the reconstructed geometry is consistent and trustworthy.


In [ ]:
frames = [
    PosedImage(
        image_path=processed_dir / f"frame_{idx:05d}.png",
        camtoworld=camtoworld[i],
        K=intrinsics[i].astype(np.float32),
    )
    for i, idx in enumerate(indices)
]

trainset = PosedImageDataset(frames, image_scale=0.25)
trainloader = DataLoader(trainset, batch_size=1, shuffle=True)
model = GaussianSplatModel(points, colors, init_scale=0.01, init_opacity=0.1).to(device)
optimizer = make_gaussian_optimizers(model, lr=1e-3)

for epoch in range(30):
    loss = train_one_epoch(model, trainloader, optimizer, device=device)
    print(f"epoch {epoch:02d} | L1 loss {loss:.4f}")

## Rendered Views, PSNR, And SSIM

After training, we render a few known camera views and compare the rendered images against the corresponding real images.

This gives us a simple quantitative check of reconstruction quality.

We use two common image metrics:

- **PSNR** measures pixel-wise reconstruction error. Higher PSNR usually means the rendered image is numerically closer to the target image.
- **SSIM** measures similarity of local image structure. It is often more aligned with perceptual image quality than raw pixel error, but it is still only a simple metric.

Both metrics are useful, but neither metric truly understands the scene. In this notebook, that matters because surgical/endoscopic data can contain difficult visual effects such as specular highlights, blur, non-Lambertian tissue appearance, deformation, smoke, fluid, and tool motion.

So treat PSNR and SSIM as helpful summaries, not as the final truth.

A good evaluation combines:

- the metric table,
- side-by-side visual comparisons,
- inspection of the point cloud,
- and common sense about whether the geometry is actually plausible.

When comparing COLMAP and DA3 initializations, look for differences such as:

- sharper or blurrier renders,
- better or worse coverage of low-texture regions,
- floating artifacts,
- missing surfaces,
- and whether the model overfits known views without producing convincing novel views.


In [ ]:
source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
deterministic_sample_names = [image_paths[idx].name for idx in source_sample_ids]
requested_names = deterministic_sample_names
print(f"Using deterministic {da3_experiment} source samples:", requested_names)

source_names = [image_paths[idx].name for idx in indices]
source_to_frame_id = {name: frame_id for frame_id, name in enumerate(source_names)}
matched_ids = [source_to_frame_id[name] for name in requested_names if name in source_to_frame_id]
if matched_ids:
    sample_ids = np.asarray(matched_ids, dtype=int)
else:
    print("Requested source samples were not available in DA3 outputs; using evenly spaced DA3 frames.")
    sample_ids = np.linspace(0, len(frames) - 1, num=min(3, len(frames)), dtype=int)
metric_rows = []

for frame_id in sample_ids:
    source_name = image_paths[indices[frame_id]].name
    print("rendering", source_name)
    rendered = render_posed_image(model, frames[frame_id], image_scale=0.25, device=device)
    metrics = image_metrics(rendered["render"], rendered["target"])
    metric_rows.append({"frame": frames[frame_id].image_path.name, "source_frame": source_name, **metrics})
    show_render_comparison(
        rendered["target"],
        rendered["render"],
        rendered["alpha"],
        metrics=metrics,
    )

display(pd.DataFrame(metric_rows))

## Render An Interpolated DA3 Trajectory GIF

Finally, we render a short GIF along an interpolated camera trajectory.

Instead of rendering only the original training views, we create virtual cameras between selected anchor frames. This is useful because novel-view rendering often reveals problems that are not obvious from training-view metrics.

For example, a reconstruction may look acceptable from the input cameras but fall apart when viewed from slightly different positions. Floating geometry, holes, incorrect depth ordering, and over-smoothed surfaces are often easier to spot in a moving render.

This section uses the same source-frame anchors as the COLMAP section, but mapped onto the DA3 camera trajectory. The coordinate systems of COLMAP and DA3 are not necessarily identical, so we do not compare raw coordinates directly. Instead, we compare the semantic camera path: we interpolate between the same original image choices and render the in-between views.

When watching the GIF, pay attention to:

- whether motion feels smooth,
- whether structures remain stable,
- whether surfaces shimmer or float,
- whether geometry disappears at certain viewpoints,
- and whether DA3 initialization improves coverage compared with the sparse COLMAP baseline.

The GIF is not just for presentation. It is a very useful debugging tool.


In [ ]:
source_sample_ids = np.linspace(0, len(image_paths) - 1, num=min(3, len(image_paths)), dtype=int)
anchor_names = [image_paths[idx].name for idx in source_sample_ids]
print(f"Using deterministic {da3_experiment} trajectory anchors:", anchor_names)

source_names = [image_paths[idx].name for idx in indices]
source_to_frame_id = {name: frame_id for frame_id, name in enumerate(source_names)}
anchor_source_names = [name for name in anchor_names if name in source_to_frame_id]
anchor_frames = [frames[source_to_frame_id[name]] for name in anchor_source_names]
if len(anchor_frames) < 2:
    raise ValueError("Need at least two DA3 anchor frames for an interpolated trajectory.")

novel_cameras = interpolate_camera_path(anchor_frames, steps_per_segment=20, include_keyframes=False)
gif_renders = [
    render_virtual_camera(
        model,
        camera["camtoworld"],
        camera["K"],
        reference_frame=anchor_frames[0],
        image_scale=1.0,
        device=device,
    )["render"]
    for camera in novel_cameras
]
gif_path = save_render_gif(stream_root / "da3_3dgs_trajectory.gif", gif_renders, fps=8)
print("anchor source frames:", anchor_source_names)
print("novel views:", len(gif_renders))
DisplayImage(filename=str(gif_path))

## Discussion

Use this section to reflect on what changed when we replaced sparse COLMAP initialization with dense DA3 initialization.

Some guiding questions:

1. **Where does DA3 provide geometry that sparse COLMAP missed?**  
   Look especially at smooth, weakly textured, blurry, or repeatedly moving regions. These are often hard for feature matching.

2. **Which DA3-filled surfaces look plausible but may not be directly supported by matching evidence?**  
   Dense learned depth can fill in surfaces that look reasonable, but that does not guarantee multi-view correctness.

3. **Why might DA3 renders from all frames look blurrier than the COLMAP baseline?**  
   Possible reasons include noisy depth, pose drift, inconsistent predictions between frames, too many uncertain points, or a harder optimization problem caused by dense but imperfect initialization.

4. **What changes in the `every_5th` setting?**  
   Compare COLMAP and DA3 when fewer frames are used. Does DA3 still provide dense coverage? Does COLMAP become much sparser? Does 3DGS benefit more from DA3 when the classical reconstruction has fewer matching opportunities?

5. **How do point clouds and final renders relate?**  
   A point cloud can look dense and impressive but still lead to poor renders if points are misplaced. Conversely, a sparse point cloud can sometimes be enough if the camera poses are accurate and the visible scene is simple.

6. **What would you try next?**  
   Some natural experiments are:
   - increase or decrease the confidence threshold,
   - change the point sampling stride,
   - reduce `max_points`,
   - train longer,
   - compare different frame subsets,
   - inspect failure cases visually rather than relying only on metrics.

The main takeaway is that initialization is a trade-off. COLMAP gives sparse geometry backed by feature matches. DA3 gives dense geometry from learned depth prediction. For 3DGS, the best starting point is not necessarily the densest one, but the one that gives the optimizer useful and consistent information.
